### Измерение положения звезды Wolf 359 по космическим снимкам

In [1]:
!{'ls fits'}

lor_0449933827_0x633_sci.fit   readme.txt
lor_0449933827_0x633_sci.fits  wolf359_20200326.fits
lor_0449933832_0x633_sci.fit   wolf359_20200425.fits
lor_0449933837_0x633_sci.fit   wolf359_cr_wcs.fits


In [2]:
from astropy.io import fits
from math import *
import numpy as np
import pandas as pd


from scipy.optimize import curve_fit

from astropy.wcs import WCS
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.time import Time
from astropy.time import TimeDelta

import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import SymLogNorm, LogNorm
from mpl_toolkits.mplot3d.axes3d import Axes3D
%matplotlib widget

hdul = fits.open('fits/lor_0449933827_0x633_sci.fits')
imdata = hdul[0].data
h,w = np.shape(imdata)
W = WCS(hdul[0].header)

In [3]:
cwolf359 = SkyCoord('10:56:23.67 +06:59:58.34', unit=(u.hourangle, u.deg))
x__wolf359, y__wolf359 = W.wcs_world2pix(np.array([[cwolf359.ra.degree,cwolf359.dec.degree]], np.float_), 1)[0]
x_wolf359, y_wolf359 = 125,123

cwolf359.ra.degree,cwolf359.dec.degree

(164.09862499999997, 6.999538888888889)

In [4]:
t = Time(hdul[0].header['SPCUTCAL'])+TimeDelta(float(hdul[0].header['EXPTIME']) / 2.0, format='sec')
JD = float(t.copy(format='jd').value)
yr = float(t.copy(format='jyear').value)
t

<Time object: scale='utc' format='isot' value=2020-04-23T07:45:09.600>

In [5]:
import ssl
from astroquery.gaia import Gaia
limmag1,limmag2 = 10,16.5
fov = 0.4
ssl._create_default_https_context = ssl._create_unverified_context
job = Gaia.launch_job_async("SELECT * \
FROM gaiadr3.gaia_source \
WHERE CONTAINS(POINT('ICRS',gaiadr3.gaia_source.ra,gaiadr3.gaia_source.dec),CIRCLE('ICRS',%f,%f,%f))=1\
                           AND  phot_g_mean_mag>%f AND  phot_g_mean_mag<%f;"%(cwolf359.ra.degree,cwolf359.dec.degree,fov,limmag1,limmag2) \
                            , dump_to_file=False)

gaiat = job.get_results()


INFO: Query finished. [astroquery.utils.tap.core]


In [6]:
eqp = np.asarray([gaiat['ra'],gaiat['dec']]).T
border = 20
pxpos = W.wcs_world2pix(eqp,1)
p_indx = np.where((pxpos[:,0]>border) & (pxpos[:,0]<w-border) & (pxpos[:,1]>border) & (pxpos[:,1]<h-border))
pixpos = pxpos[p_indx]
ep = eqp[p_indx]
ap = 11

In [7]:
# рисуем кадр и помечаем на нем звезды
cf1,cf2 = 0.5,3.0# определяет окно значений отсчетов, которые будут отрисованы (смотри plt.imshow дальше в этом блоке) 
# определяет как рисовать систему координат. 
#True - использует WCS и отображаются экваториальные координаты 
#False - отображаются пиксельные координаты 
wcs_based = False
if wcs_based:
    fig, ax = plt.subplots(figsize=(2.6,2.6), dpi=300, tight_layout=True, subplot_kw={'projection': W})
    ax.coords[0].set_ticklabel(size=4)
    ax.coords[1].set_ticklabel(size=4)
    plt.xlabel('RA', fontsize = 4)
    plt.ylabel('DEC', fontsize=4)
else:
    fig, ax = plt.subplots(figsize=(2.6,2.6), dpi=300)
    ax.xaxis.set_tick_params(labelsize=5)
    ax.yaxis.set_tick_params(labelsize=5)
    plt.xlabel('X, pix', fontsize = 4)
    plt.ylabel('Y, pix', fontsize=4)
#рисуем изображение в серой шкале. Можно поменять cmap='gray' на что-нибудь еще и можно получить всякие более модные отображения
plt.imshow(imdata, vmin=np.median(imdata)- cf1*np.std(imdata) ,
           vmax=np.median(imdata)  + cf2*np.std(imdata), origin='lower', cmap='gray')
#рисуем метки звезд согласно WCS и координатам звезд из Gaia DR3 
plt.scatter(pixpos[:,0], pixpos[:,1], s=ap/3, facecolors='none', edgecolors='g', linewidth=0.2)
plt.scatter(x__wolf359, y__wolf359, s=ap/3, facecolors='none', edgecolors='y', linewidth=0.2)
plt.scatter(x_wolf359, y_wolf359, s=ap/3, facecolors='none', edgecolors='r', linewidth=0.2)

#автоматически подгоняем рамку x_wolf359, y_wolf359
plt.tight_layout()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [8]:
# функции преобразования координат
def tangential_coordinates(ra,dec,RA,DEC):
    ksi = cos(dec)*sin(ra-RA)/(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))
    eta = (sin(dec)*cos(DEC)-cos(dec)*sin(DEC)*cos(ra-RA))/(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))
    return ksi,eta

def equatorial_coordinates(ksi,eta,RA,Dec):
    x,y,z = np.dot(np.array([[-sin(RA),-cos(RA) * sin(Dec),cos(RA) * cos(Dec)],
                             [cos(RA),-sin(RA) * sin(Dec),sin(RA) * cos(Dec)],
                             [0,cos(Dec),sin(Dec)]]),
                   np.array([ksi,eta,1]))/sqrt(1+ksi*ksi+eta*eta)
    ra = atan2(y,x)
    dec = atan2(z,sqrt(x*x+y*y))
    if(ra<0):
        ra+=2*pi
    return ra,dec

def plot_img3D(I,el,az,text):
    box = np.shape(I)[0]
    X = Y = np.arange(0, box, 1) 
    X, Y = np.meshgrid(X, Y)
    fig = plt.figure(figsize=(2.0,2.0), dpi=150, tight_layout=True)
    ax = fig.gca(projection='3d')
    ax.plot_surface(X, Y, I, rstride=1, cstride=1, cmap=cm.Greys, \
                    linewidth=0, antialiased=True)
    ax.view_init(elev=el, azim=az)
    ax.set_xlabel('x', fontsize=5)
    ax.set_ylabel('y', fontsize=5)
    ax.set_title(text, fontsize=5)
    ax.xaxis.set_tick_params(labelsize=5)
    ax.yaxis.set_tick_params(labelsize=5)
    ax.zaxis.set_tick_params(labelsize=5)
    
# уровень фона
def im_bkgr(I):
    Np =  np.size(I)
    N = int(Np/3)
    B = np.median(I)
    while Np>N:
        J = I[np.where(I<B)]
        B = np.median(J)
        Np = np.size(J)
    return B

def center_mass(I):
    Np =  np.size(I)
    N = int(Np/3)
    B = np.median(I)
    while Np>N:
        J = I[np.where(I<B)]
        B = np.median(J)
        Np = np.size(J)
    box = np.shape(I)[0]
    X = Y = np.arange(0, box, 1) 
    X, Y = np.meshgrid(X, Y)
    I = np.ravel(I) - B
    X = np.ravel(X)
    Y = np.ravel(Y)
    return np.dot(X,I)/np.sum(I),np.dot(Y,I)/np.sum(I),B, I.max()

# строим нашу функцию для оптимизации. Это просто гауссоида, которая была выше описана
def im_profile(box,*p):
    x = np.arange(0, box, 1)
    y = np.arange(0, box, 1)
    x, y = np.meshgrid(x, y)
    r = p[2]*(x-p[5])**2+p[3]*(y-p[6])**2+p[4]*(x-p[5])*(y-p[6])
    return np.ravel(p[0]+p[1]*np.exp(-r))
    

In [9]:
x,y = int(pixpos[5,0]),int(pixpos[5,1])
I = imdata[y-ap:y+ap,x-ap:x+ap]
ymax,xmax = np.unravel_index(I.argmax(), I.shape)
# print(xmax,ymax)
xc,yc,Ibkg,Im = center_mass(I)
p,covp = curve_fit(f=im_profile, p0=np.array([Ibkg,Im,1.0,1.0,0,xc,yc]),\
                   xdata = 2*ap,\
                   ydata=I.ravel())
Ibkg,Im,A,B,C,xc,yc = p
print(p)
plot_img3D(I-np.reshape(im_profile(2*ap,*p),(2*ap,2*ap)),45,45,'xc = %7.3f, yc = %7.3f'%(xc,yc))

[24.0091138  69.1520095   1.74700246  1.49535405 -0.59846349 11.49309824
 10.62783923]


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:23: MatplotlibDeprecationWarning: Calling gca() with keyword arguments was deprecated in Matplotlib 3.4. Starting two minor releases later, gca() will take no keyword arguments. The gca() function should only be used to get the current axes, or if no axes exist, create new axes with default keyword arguments. To create a new axes with non-default arguments, use plt.axes() or plt.subplot().


In [10]:
x,y= np.array([]),np.array([])
ksi,eta= np.array([]),np.array([])
for k,pix in enumerate(pixpos):
    xp,yp = int(pix[0]),int(pix[1])
    try:
        I = imdata[yp-ap:yp+ap,xp-ap:xp+ap]
        xc,yc,Ibkg,Im = center_mass(I) 
        p,covp = curve_fit(f=im_profile, p0=np.array([Ibkg,Im,1.0,1.0,0,xc,yc]),\
                       xdata = 2*ap,\
                       ydata=I.ravel())
        
        x=np.append(x,xp-ap+p[5])
        y=np.append(y,yp-ap+p[6])
        u,v = tangential_coordinates(
            radians(ep[k,0]),
            radians(ep[k,1]),
            radians(cwolf359.ra.degree),
            radians(cwolf359.dec.degree)
        )
        ksi=np.append(ksi,3600*degrees(u))
        eta=np.append(eta,3600*degrees(v))
    except Exception:
        continue

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:67: RuntimeWarning: overflow encountered in exp


In [11]:
def calibration(x,y,ksi,eta,n,Nmin,rmax):
    Q = int((n+1)*(n+2)/2)
    C = np.ones((np.size(x), Q))
    q = 0
    for i in range(n + 1):
        for j in range(n + 1):
            if (i + j <= n):
                C[:, q] = (x ** i) * (y ** j)
                q += 1
    We = np.diag(np.ones(np.size(x)))
    flag = 0
    it = 0
    while (flag == 0):
        Zx = np.dot(np.linalg.inv(np.dot(np.dot(C.T, We), C)), np.dot(np.dot(C.T, We), ksi))
        Zy = np.dot(np.linalg.inv(np.dot(np.dot(C.T, We), C)), np.dot(np.dot(C.T, We), eta))
        rx = np.dot(We, ksi - np.dot(C, Zx))
        ry = np.dot(We, eta - np.dot(C, Zy))
        r = np.sqrt(rx ** 2 + ry ** 2)
        kmax = np.argmax(r)
        flag = 1
        if (np.size(ksi) - it - Q <= Nmin):
            break
        if (r[kmax] > rmax):
            We[kmax, kmax] = 0
            flag = 0
            it += 1
    uwex = np.dot(np.dot(np.transpose(rx), We), rx) / (np.size(x) - it - Q)
    uwey = np.dot(np.dot(np.transpose(ry), We), ry) / (np.size(y) - it - Q)
    return Zx,Zy,sqrt(uwex),sqrt(uwey),np.size(x)-it

def transform(x,y,Zx):
    Q = np.size(Zx)
    n = int(np.max(np.roots(np.array([1,3,-2*(Q-1)]))))
    # print(n)
    C = np.ones((np.size(x), Q))
    q = 0
    for i in range(n+1):
        for j in range(n+1):
            if (i + j <= n):
                C[:, q] = (x ** i) * (y ** j)
                q += 1
    return np.dot(C,Zx)

In [12]:
n = 1
Q = int((n + 1) * (n + 2) / 2)
Zx, Zy, uwex, uwey, nrs = calibration(x, y, ksi, eta, n, Q + 3, 0.25)
r_ksi,r_eta = ksi - transform(x,y,Zx),eta - transform(x,y,Zy)

print(' uw errors = %6.4f, %6.4f,N = %d, Nu = %d, Q = %d' % ( uwex, uwey, np.size(ksi), nrs, Q))

# r_ksi,r_eta

 uw errors = 0.2345, 0.1795,N = 27, Nu = 9, Q = 3


In [13]:
xp,yp = int(x_wolf359),int(y_wolf359)
I = imdata[yp-ap:yp+ap,xp-ap:xp+ap]
xc,yc,Ibkg,Im = center_mass(I) 
p,covp = curve_fit(f=im_profile, p0=np.array([Ibkg,Im,1/3,1/3,0,xc,yc]),\
               xdata = 2*ap,\
               ydata=I.ravel())

X_wolf359,Y_wolf359 = xp-ap+p[5],yp-ap+p[6]
u,v = transform(X_wolf359,Y_wolf359,Zx),transform(X_wolf359,Y_wolf359,Zy)
RA_wolf359,DEC_wolf359 = np.degrees(equatorial_coordinates(
    radians(u/3600.0),
    radians(v/3600.0),
    radians(cwolf359.ra.degree),
    radians(cwolf359.dec.degree)))

C_wolf359 = SkyCoord(RA_wolf359,DEC_wolf359, frame='icrs', unit='deg')
C_wolf359.to_string('hmsdms', sep=':', precision=3)

'10:56:22.641 +07:00:03.673'

In [14]:
cwolf359.to_string('hmsdms', sep=':', precision=3)

'10:56:23.670 +06:59:58.340'